# 📈 Stock Bot - Embedding Model Training (Colab Pro)

Google Colab Pro 환경에서 임베딩 모델을 훈련하는 노트북입니다.

## 🎯 주요 기능
- **대조 학습(Contrastive Learning)**: 시간적으로 가까운 샘플은 유사하게, 먼 샘플은 다르게 임베딩
- **GPU 가속**: Colab Pro의 고성능 GPU 활용 (V100, A100)
- **Google Drive 연동**: 데이터셋 및 모델 체크포인트 저장
- **TensorBoard 시각화**: 실시간 훈련 모니터링

## 📋 사전 준비
1. Google Drive에 데이터셋 업로드: `내 드라이브/ColabData/datasets/stockbot/datasets_norm_all.duckdb`
2. Colab Pro 구독 (고성능 GPU 사용)
3. 런타임 유형: GPU (V100 또는 A100 권장)

## 🚀 훈련 전략
- **초기 훈련**: 30 epochs, learning rate 1e-4
- **Fine-tuning**: 15 epochs, learning rate 5e-5
- **배치 크기**: 256 (GPU 메모리에 따라 조정)
- **최대 샘플**: 500만 개 (메모리 제한)

## 1️⃣ 환경 설정

In [ ]:
# GPU 확인
!nvidia-smi

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 필수 패키지 설치
!pip install -q duckdb torch torchvision tensorboard scikit-learn numpy pandas

In [ ]:
# GitHub 저장소 클론 (프로젝트 코드)
import os

# 이미 클론되어 있으면 스킵
if not os.path.exists('/content/stock-bot2'):
    !git clone https://github.com/YOUR_USERNAME/stock-bot2.git /content/stock-bot2
else:
    print("Repository already cloned")

# 작업 디렉토리 변경
%cd /content/stock-bot2

## 2️⃣ 설정 및 경로 지정

In [ ]:
import os
from pathlib import Path

# ========================================
# 경로 설정 (사용자 환경에 맞게 수정)
# ========================================

# Google Drive 경로
DRIVE_ROOT = "/content/drive/MyDrive/ColabData"

# 데이터베이스 경로
DB_PATH = f"{DRIVE_ROOT}/datasets/stockbot/datasets_norm_all.duckdb"

# 출력 디렉토리 (모델 체크포인트 저장)
OUTPUT_BASE = f"{DRIVE_ROOT}/models/stockbot/embedding"

# 로컬 작업 디렉토리 (빠른 I/O를 위해)
LOCAL_WORK_DIR = "/content/work"

# 디렉토리 생성
os.makedirs(OUTPUT_BASE, exist_ok=True)
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

print(f"✓ Database: {DB_PATH}")
print(f"✓ Output: {OUTPUT_BASE}")
print(f"✓ Work dir: {LOCAL_WORK_DIR}")

# 데이터베이스 파일 존재 확인
if os.path.exists(DB_PATH):
    print(f"✓ Database file found: {os.path.getsize(DB_PATH) / 1e9:.2f} GB")
else:
    print(f"⚠️  Database file not found!")
    print(f"   Please upload datasets_norm_all.duckdb to:")
    print(f"   {DB_PATH}")

In [ ]:
# ========================================
# 훈련 설정
# ========================================

# 데이터 필터링
YEAR = 2024
START_MONTH = 9
END_MONTH = 12

# 모델 하이퍼파라미터
SEQ_LEN = 60
EMBEDDING_DIM = 128
NUM_HEADS = 4

# 훈련 하이퍼파라미터
BATCH_SIZE = 256  # GPU 메모리에 따라 조정 (128, 256, 512)
INITIAL_EPOCHS = 30  # 초기 훈련 에포크
FINETUNE_EPOCHS = 15  # Fine-tuning 에포크
INITIAL_LR = 1e-4  # 초기 학습률
FINETUNE_LR = 5e-5  # Fine-tuning 학습률

# 대조 학습 설정
TEMPERATURE = 0.07
POSITIVE_TIME_THRESHOLD = 10  # 초
NEGATIVE_TIME_THRESHOLD = 60  # 초

# 데이터 로더 설정
NUM_WORKERS = 2  # Colab에서는 2-4 권장
MAX_SAMPLES = 5000000  # 500만 샘플 (메모리 제한)

# 체크포인트 설정
CHECKPOINT_INTERVAL = 5  # 에포크
VAL_EVERY = 3  # 검증 주기

print("✓ Training configuration loaded")

## 3️⃣ 월별 훈련 실행

In [ ]:
# 필요한 모듈 임포트
import sys
sys.path.insert(0, '/content/stock-bot2')

import torch
import logging
from datetime import datetime

# 로깅 설정
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# GPU 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n{'='*80}")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"{'='*80}\n")

In [ ]:
# 월별 날짜 계산 함수
def get_month_dates(year, month):
    """
    월의 시작일과 종료일 계산
    
    Args:
        year: 연도
        month: 월
        
    Returns:
        (start_date, end_date) 튜플 (YYYYMMDD 형식)
    """
    from datetime import datetime, timedelta
    
    # 시작일
    start_date = f"{year:04d}{month:02d}01"
    
    # 종료일 (다음 달 1일 - 1일)
    if month == 12:
        end_year = year + 1
        end_month = 1
    else:
        end_year = year
        end_month = month + 1
    
    # 마지막 날 계산
    next_month = datetime(end_year, end_month, 1)
    last_day = next_month - timedelta(days=1)
    end_date = last_day.strftime("%Y%m%d")
    
    return start_date, end_date

# 테스트
start, end = get_month_dates(2024, 9)
print(f"Example: 2024년 9월 -> {start} ~ {end}")

In [ ]:
# 초기 훈련 함수
def train_initial(year, month):
    """
    초기 훈련 (첫 번째 달)
    
    Args:
        year: 연도
        month: 월
        
    Returns:
        체크포인트 경로
    """
    start_date, end_date = get_month_dates(year, month)
    output_dir = f"{OUTPUT_BASE}_{year}_{month:02d}"
    
    print(f"\n{'='*80}")
    print(f"초기 훈련: {year}년 {month}월")
    print(f"날짜: {start_date} ~ {end_date}")
    print(f"출력: {output_dir}")
    print(f"{'='*80}\n")
    
    # 훈련 명령 실행
    cmd = f"""
    python -m ai_trader.embedding.train_embedding \
        --db "{DB_PATH}" \
        --table datasets \
        --out "{output_dir}" \
        --seq-len {SEQ_LEN} \
        --embedding-dim {EMBEDDING_DIM} \
        --batch-size {BATCH_SIZE} \
        --epochs {INITIAL_EPOCHS} \
        --lr {INITIAL_LR} \
        --device cuda \
        --positive-time-threshold {POSITIVE_TIME_THRESHOLD} \
        --negative-time-threshold {NEGATIVE_TIME_THRESHOLD} \
        --temperature {TEMPERATURE} \
        --num-heads {NUM_HEADS} \
        --val-every {VAL_EVERY} \
        --checkpoint-interval {CHECKPOINT_INTERVAL} \
        --start-date {start_date} \
        --end-date {end_date} \
        --num-workers {NUM_WORKERS} \
        --max-samples {MAX_SAMPLES}
    """
    
    !{cmd}
    
    checkpoint_path = f"{output_dir}/checkpoint_epoch{INITIAL_EPOCHS}.pt"
    print(f"\n✓ {year}년 {month}월 초기 훈련 완료")
    print(f"✓ Checkpoint: {checkpoint_path}\n")
    
    return checkpoint_path

In [ ]:
# Fine-tuning 함수
def train_finetune(year, month, prev_checkpoint):
    """
    Fine-tuning (이후 달)
    
    Args:
        year: 연도
        month: 월
        prev_checkpoint: 이전 체크포인트 경로
        
    Returns:
        체크포인트 경로
    """
    start_date, end_date = get_month_dates(year, month)
    output_dir = f"{OUTPUT_BASE}_{year}_{month:02d}"
    
    print(f"\n{'='*80}")
    print(f"Fine-tuning: {year}년 {month}월")
    print(f"날짜: {start_date} ~ {end_date}")
    print(f"이전 모델: {prev_checkpoint}")
    print(f"출력: {output_dir}")
    print(f"{'='*80}\n")
    
    # 훈련 명령 실행
    cmd = f"""
    python -m ai_trader.embedding.train_embedding \
        --db "{DB_PATH}" \
        --table datasets \
        --out "{output_dir}" \
        --seq-len {SEQ_LEN} \
        --embedding-dim {EMBEDDING_DIM} \
        --batch-size {BATCH_SIZE} \
        --epochs {FINETUNE_EPOCHS} \
        --lr {FINETUNE_LR} \
        --device cuda \
        --positive-time-threshold {POSITIVE_TIME_THRESHOLD} \
        --negative-time-threshold {NEGATIVE_TIME_THRESHOLD} \
        --temperature {TEMPERATURE} \
        --num-heads {NUM_HEADS} \
        --val-every {VAL_EVERY} \
        --checkpoint-interval {CHECKPOINT_INTERVAL} \
        --start-date {start_date} \
        --end-date {end_date} \
        --num-workers {NUM_WORKERS} \
        --max-samples {MAX_SAMPLES} \
        --resume "{prev_checkpoint}"
    """
    
    !{cmd}
    
    checkpoint_path = f"{output_dir}/checkpoint_epoch{FINETUNE_EPOCHS}.pt"
    print(f"\n✓ {year}년 {month}월 Fine-tuning 완료")
    print(f"✓ Checkpoint: {checkpoint_path}\n")
    
    return checkpoint_path

In [ ]:
# 월별 순차 훈련 실행
print(f"\n{'='*80}")
print(f"월별 자동 훈련 시작")
print(f"{'='*80}")
print(f"기간: {YEAR}년 {START_MONTH}월 ~ {END_MONTH}월")
print(f"배치 크기: {BATCH_SIZE}")
print(f"최대 샘플: {MAX_SAMPLES:,}")
print(f"{'='*80}\n")

prev_checkpoint = None
trained_models = []

for month in range(START_MONTH, END_MONTH + 1):
    if prev_checkpoint is None:
        # 첫 달: 초기 훈련
        checkpoint = train_initial(YEAR, month)
        prev_checkpoint = checkpoint
    else:
        # 이후 달: Fine-tuning
        checkpoint = train_finetune(YEAR, month, prev_checkpoint)
        prev_checkpoint = checkpoint
    
    trained_models.append(checkpoint)

print(f"\n{'='*80}")
print("✓ 전체 월별 훈련 완료!")
print(f"{'='*80}")
print("훈련된 모델:")
for model_path in trained_models:
    print(f"  - {model_path}")
print(f"{'='*80}\n")

## 4️⃣ TensorBoard 시각화

In [ ]:
# TensorBoard 로드
%load_ext tensorboard

In [ ]:
# TensorBoard 실행 (특정 월 선택)
# 예: 2024년 9월 모델의 로그 보기
month_to_view = 9
tensorboard_logdir = f"{OUTPUT_BASE}_{YEAR}_{month_to_view:02d}/tensorboard_logs"

%tensorboard --logdir {tensorboard_logdir}

In [ ]:
# 모든 월의 TensorBoard 로그 보기 (비교)
%tensorboard --logdir {OUTPUT_BASE}

## 5️⃣ 모델 평가

In [ ]:
# 특정 월의 모델 평가
month_to_eval = 12  # 평가할 월 선택
model_path = f"{OUTPUT_BASE}_{YEAR}_{month_to_eval:02d}/checkpoint_epoch{FINETUNE_EPOCHS if month_to_eval > START_MONTH else INITIAL_EPOCHS}.pt"

print(f"\n{'='*80}")
print(f"모델 평가: {YEAR}년 {month_to_eval}월")
print(f"모델: {model_path}")
print(f"{'='*80}\n")

# 평가 명령 실행
!python -m ai_trader.embedding.evaluate_embedding \
    --model "{model_path}" \
    --db "{DB_PATH}" \
    --table datasets \
    --device cuda

## 6️⃣ 모델 다운로드 (선택사항)

In [ ]:
# 훈련된 모델을 압축하여 다운로드
import shutil
from datetime import datetime

# 압축 파일 이름
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"/content/embedding_models_{YEAR}_{START_MONTH:02d}_{END_MONTH:02d}_{timestamp}"

# 모델 디렉토리 압축
shutil.make_archive(zip_filename, 'zip', OUTPUT_BASE)

print(f"✓ Models zipped: {zip_filename}.zip")
print(f"✓ File size: {os.path.getsize(zip_filename + '.zip') / 1e9:.2f} GB")

In [ ]:
# 로컬로 다운로드
from google.colab import files

# 주의: 파일 크기가 크면 시간이 오래 걸릴 수 있습니다
# Google Drive에 이미 저장되어 있으므로 필요시에만 실행
# files.download(zip_filename + '.zip')

print("모델은 이미 Google Drive에 저장되어 있습니다.")
print(f"경로: {OUTPUT_BASE}")

## 💡 유용한 팁

### GPU 메모리 부족 시
```python
# 배치 크기 줄이기
BATCH_SIZE = 128  # 또는 64

# 임베딩 차원 줄이기
EMBEDDING_DIM = 64  # 기본값: 128
```

### 훈련 속도 향상
```python
# 검증 주기 늘리기
VAL_EVERY = 5  # 기본값: 3

# 워커 수 조정
NUM_WORKERS = 4  # 기본값: 2
```

### 데이터 샘플 수 조정
```python
# 더 많은 샘플 사용 (메모리 충분 시)
MAX_SAMPLES = 10000000  # 1천만 샘플

# 더 적은 샘플 사용 (빠른 테스트)
MAX_SAMPLES = 1000000  # 100만 샘플
```

### Colab 세션 유지
- Colab Pro는 최대 24시간 연속 실행 가능
- 중간에 체크포인트가 저장되므로 중단 시 재개 가능
- Google Drive에 자동 저장되므로 안전

### 모니터링
- TensorBoard로 실시간 손실 확인
- GPU 사용률: `!nvidia-smi` 실행
- 메모리 사용: `!free -h` 실행

## 🔧 트러블슈팅

### 1. GPU OOM (Out of Memory)
```python
# 배치 크기 줄이기
BATCH_SIZE = 64  # 128 → 64

# 또는 임베딩 차원 줄이기
EMBEDDING_DIM = 64  # 128 → 64
```

### 2. 데이터베이스 연결 오류
```python
# Google Drive 경로 확인
!ls -lh /content/drive/MyDrive/ColabData/datasets/stockbot/

# 데이터베이스 파일 존재 확인
import os
print(f"DB exists: {os.path.exists(DB_PATH)}")
```

### 3. 훈련 중단 후 재개
```python
# 마지막 체크포인트 찾기
import glob
checkpoints = glob.glob(f"{OUTPUT_BASE}_2024_09/*.pt")
latest_checkpoint = max(checkpoints, key=os.path.getctime)
print(f"Latest checkpoint: {latest_checkpoint}")

# 해당 체크포인트에서 재개
# --resume 옵션에 경로 지정
```

### 4. 느린 훈련 속도
```python
# GPU 사용 확인
!nvidia-smi

# 데이터 로더 워커 수 조정
NUM_WORKERS = 4  # 2 → 4

# 검증 주기 늘리기
VAL_EVERY = 5  # 3 → 5
```